# 🌌 Galaxy Morphology Classification
### AI 100 — Deep Learning Midterm Project
**Author:** Ava Stephens  
**Dataset:** Galaxy10 DECals (10 classes, ~17,736 images)  
**Model:** Transfer Learning with ResNet18 (PyTorch)  

---
This notebook classifies galaxy images into 10 morphological categories using a pretrained ResNet18 CNN with fine-tuning.

## 1. Install & Import Dependencies

In [ ]:
# Install required libraries (run once in Colab)
!pip install h5py torch torchvision matplotlib scikit-learn seaborn tqdm --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Download the Galaxy10 DECals Dataset

In [ ]:
# Download Galaxy10 DECals dataset (~800MB HDF5 file)
import urllib.request
import os

url = 'https://astro.utoronto.ca/~hleung/shared/Galaxy10/Galaxy10_DECals.h5'
filename = 'Galaxy10_DECals.h5'

if not os.path.exists(filename):
    print('Downloading Galaxy10 DECals dataset...')
    urllib.request.urlretrieve(url, filename)
    print('Download complete!')
else:
    print('Dataset already downloaded.')

## 3. Explore the Dataset

In [ ]:
# Galaxy10 DECals class names
CLASS_NAMES = [
    'Disturbed',
    'Merging',
    'Round Smooth',
    'In-between Round Smooth',
    'Cigar Shaped Smooth',
    'Barred Spiral',
    'Unbarred Tight Spiral',
    'Unbarred Loose Spiral',
    'Edge-on without Bulge',
    'Edge-on with Bulge'
]

# Load data
with h5py.File('Galaxy10_DECals.h5', 'r') as f:
    images = np.array(f['images'])   # shape: (N, 256, 256, 3), uint8
    labels = np.array(f['ans'])      # shape: (N,), int

print(f'Total images: {len(images)}')
print(f'Image shape:  {images[0].shape}')
print(f'Classes:      {len(CLASS_NAMES)}')
print(f'Label distribution:')
for i, name in enumerate(CLASS_NAMES):
    count = np.sum(labels == i)
    print(f'  [{i}] {name}: {count}')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Galaxy10 DECals — Sample Images by Class', fontsize=14, fontweight='bold')

for cls_idx in range(10):
    row, col = divmod(cls_idx, 5)
    idx = np.where(labels == cls_idx)[0][0]
    axes[row, col].imshow(images[idx])
    axes[row, col].set_title(CLASS_NAMES[cls_idx], fontsize=8)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sample_images.png')

In [ ]:
# Class distribution bar chart
counts = [np.sum(labels == i) for i in range(10)]
plt.figure(figsize=(12, 4))
bars = plt.bar(range(10), counts, color='steelblue', edgecolor='white')
plt.xticks(range(10), [f'{i}\n{n}' for i, n in enumerate(CLASS_NAMES)], fontsize=7)
plt.ylabel('Number of Images')
plt.title('Class Distribution in Galaxy10 DECals')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Dataset & DataLoader

In [ ]:
class GalaxyDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]  # (H, W, 3) uint8
        from PIL import Image
        img = Image.fromarray(img)
        if self.transform:
            img = self.transform(img)
        label = int(self.labels[idx])
        return img, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Create full dataset and split 80/10/10
full_dataset = GalaxyDataset(images, labels, transform=None)
n = len(full_dataset)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)
n_test  = n - n_train - n_val

train_idx, val_idx, test_idx = random_split(
    range(n), [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_imgs = images[list(train_idx)]
train_labs = labels[list(train_idx)]
val_imgs   = images[list(val_idx)]
val_labs   = labels[list(val_idx)]
test_imgs  = images[list(test_idx)]
test_labs  = labels[list(test_idx)]

train_ds = GalaxyDataset(train_imgs, train_labs, train_transform)
val_ds   = GalaxyDataset(val_imgs,   val_labs,   val_transform)
test_ds  = GalaxyDataset(test_imgs,  test_labs,  val_transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False, num_workers=2)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

## 5. Build the Model — ResNet18 with Transfer Learning

In [ ]:
# Load pretrained ResNet18 and replace the final layer for 10 classes
model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all layers first (feature extraction)
for param in model.parameters():
    param.requires_grad = False

# Replace final FC layer — this one is trainable
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(num_features, 10)
)

model = model.to(device)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,} total')

## 6. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

EPOCHS = 15
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

best_val_acc = 0.0

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, lbls in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]', leave=False):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == lbls).sum().item()
        total   += imgs.size(0)

    train_loss = running_loss / total
    train_acc  = correct / total

    # ---- VALIDATE ----
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            running_loss += loss.item() * imgs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == lbls).sum().item()
            total   += imgs.size(0)

    val_loss = running_loss / total
    val_acc  = correct / total

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    print(f'Epoch {epoch+1:2d}/{EPOCHS} | '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}   | Val Acc: {val_acc:.4f}')

print(f'\nBest Validation Accuracy: {best_val_acc:.4f}')

## 7. Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, EPOCHS + 1)

ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(epochs, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
ax1.set_title('Loss over Epochs'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=4)
ax2.plot(epochs, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc',   markersize=4)
ax2.set_title('Accuracy over Epochs'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Galaxy Classifier — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluation on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbls.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = (all_preds == all_labels).mean()
print(f'Test Accuracy: {test_acc*100:.2f}%\n')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Visualize Predictions

In [ ]:
# Show 10 random test images with predicted vs true labels
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Sample Predictions (Green = Correct, Red = Wrong)', fontsize=12, fontweight='bold')

rand_indices = np.random.choice(len(test_imgs), 10, replace=False)
from PIL import Image as PILImage

for ax, idx in zip(axes.flat, rand_indices):
    img = test_imgs[idx]
    true_label = int(test_labs[idx])

    # Run inference
    pil_img = PILImage.fromarray(img)
    tensor  = val_transform(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        out  = model(tensor)
        pred = int(torch.argmax(out, 1).item())

    color = 'green' if pred == true_label else 'red'
    ax.imshow(img)
    ax.set_title(f'P: {CLASS_NAMES[pred]}\nT: {CLASS_NAMES[true_label]}',
                 fontsize=6.5, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Lessons & Experience

### What I Learned

**Transfer Learning is powerful.**  
Using a ResNet18 pretrained on ImageNet allowed the model to leverage low-level features (edges, textures) that generalize well to astronomical images — even though galaxies look nothing like everyday photos. Fine-tuning only the final layer was surprisingly effective.

**Data augmentation matters.**  
Galaxies can appear rotated at any angle, so adding random flips and rotations helped the model generalize much better than training without augmentation.

**Class imbalance is real.**  
Some galaxy classes (like Merging) have far fewer samples than others (like Round Smooth). This shows up clearly in the classification report — rarer classes tend to have lower recall.

**GPU vs. CPU training difference is huge.**  
Training on a free Colab T4 GPU took ~5 minutes per epoch. On CPU it would have taken hours. Cloud compute is essential for deep learning.

### What I Would Do Next
- Unfreeze more ResNet layers for full fine-tuning
- Address class imbalance with weighted sampling or loss weighting
- Try EfficientNet-B0 or Vision Transformers
- Apply Grad-CAM to visualize which parts of each galaxy the model focuses on